In [1]:
!pip3 install --force-reinstall "numpy<2.0"
!pip3 install pandas matplotlib


DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
  Using cached numpy-1.26.4-cp39-cp39-macosx_10_9_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp39-cp39-macosx_10_9_x86_64.whl (20.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, 

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime

In [3]:
# Setting up colors for graphs

COLORS = {
	'Running':           '#f97316',
	'Treadmill Running': '#fb923c',
	'Bouldering':        '#a78bfa',
	'Strength Training': '#34d399',
	'Yoga':              '#60a5fa', # Stretching
	'Hiking':            '#fbbf24',
	'Other':             '#9ca3af',
}

I purchased my Garmin Forerunner 265 on 12/1/24, and have been logging workouts daily since then.

Recently, I discovered that Garmin allows you to easily export activities on a rolling six month window.

While the watch provides basic insights on a week-to-week basis, I wanted to dive a little deeper into how my habits have changed. I've slowly shifted away from climbing as the weather on the East Coast has started to get warmer, and been on a 4-5 day lifting schedule. I'm curious to see how my body has held up.

In [5]:
df_raw = pd.read_csv("activities.csv")

# Parsing helpers
def parse_duration(s):
	if pd.isna(s) or s == '--':
		return np.nan
	s = str(s).strip()
	p = s.split(":")
	try:
		if len(p) == 3:
			return int(p[0])*3600 + int(p[1])*60 + float(p[2])
		elif len(p) == 2:
			return int(p[0])*60 + float(p[1])
		else:
			return float(p[0])
	except IndexError:
		return np.nan

def clean(s):
	if pd.isna(s) or str(s).strip() in ('--', ''):
		return np.nan
	return pd.to_numeric(str(s).replace(',', ''), errors='coerce')

In [8]:
# Some janitor/plumbing work:

df = df_raw.copy()
df['Date'] = pd.to_datetime(df['Date'])
df['date_only'] = df['Date'].dt.date
df['weekday'] = df['Date'].dt.day_name()
df['weekday_num'] = df['Date'].dt.dayofweek  # 0=Mon ... 6=Sun
df['hour'] = df['Date'].dt.hour
df['week'] = df['Date'].dt.to_period('W')
df['month'] = df['Date'].dt.to_period('M')
df['month_label'] = df['Date'].dt.strftime('%b %Y')

# Duration in minutes
df['duration_sec'] = df['Time'].apply(parse_duration)
df['duration_min'] = df['duration_sec'] / 60

for col in ['Calories', 'Avg HR', 'Max HR', 'Aerobic TE', 'Distance',
			'Total Reps', 'Total Sets', 'Total Routes']:
	df[col] = df[col].apply(clean)

# Specifics for bouldering
df['climb_time_sec'] = df['Climb Time'].apply(parse_duration)
df['rest_sec'] = df['Total Rest'].apply(parse_duration)
df['climb_ratio'] = df['climb_time_sec'] / df['duration_sec']

df['pace_sec_mi'] = df['Avg Pace'].apply(parse_duration)

# Tidy activity type labels
def simplify(t):
	if 'Running' in str(t):
		return t  # keep Treadmill vs Outdoor
	return t

df['activity_group'] = df['Activity Type'].apply(
	lambda x: x if x in COLORS else 'Other'
)

In [9]:
print(f'Loaded {len(df)} activities from {df["Date"].min().date()} to {df["Date"].max().date()}')
df[['Date','Activity Type','duration_min','Calories','Avg HR']].head(8)

Loaded 160 activities from 2025-10-12 to 2026-03-21


,Date,Activity Type,duration_min,Calories,Avg HR
0,2026-03-21 09:19:08,Running,43.650000,487,143.0
1,2026-03-20 07:42:13,Strength Training,32.000000,156,99.0
2,2026-03-19 19:04:26,Bouldering,98.866667,510,107.0
3,2026-03-19 07:11:47,Strength Training,43.983333,226,102.0
4,2026-03-17 19:29:08,Treadmill Running,41.916667,508,149.0
5,2026-03-17 06:53:49,Strength Training,69.900000,346,103.0
6,2026-03-16 07:11:29,Strength Training,60.783333,352,108.0
7,2026-03-13 17:42:40,Strength Training,18.600000,133,116.0
